# Transfer learning v2 — fine-tuning completo, costo de errores y explicabilidad

**Proyecto:** IC522 · Trabajo Final Integrador · **Alumno:** WLODECK, Fabricio Joaquín
**Archivo:** `02b_transfer_learning_mandioca_v2.ipynb` — complementa (no reemplaza) a `02_transfer_learning_mandioca.ipynb`.
**Datos:** **mismo 20 % que la v1** → `datos\splits\manifest_20pct.csv` **se lee, no se regenera**: split idéntico, así los resultados de v1 siguen siendo comparables y los checkpoints/embeddings existentes se reutilizan.

## Qué corrige esta versión (crítica recibida sobre la v1)

| # | Problema señalado | Qué hace la v2 |
|---|---|---|
| 1 | E1-E4 entrenaban ~6 k parámetros sobre backbone congelado → macro-F1 0,58-0,63 era lo esperable | **Fine-tuning completo** de MobileNetV2: AdamW, **lr diferencial** (backbone 1e-4 / cabeza 1e-3), **cosine + warmup**, **label smoothing**, AMP, `channels_last` |
| 2 | `Resize(800×600→256) + CenterCrop(224)` perdía ~34 % del ancho; `RandomCrop(224)` podía sacar la lesión del cuadro | **`Resize(384,384)` directo, sin crop** (deformación 4:3→1:1 leve, una CNN con GAP acepta entradas rectangulares) |
| 3 | "Aug no ayudó" estaba confundido con el backbone congelado; E5 hacía FT **sin** aug | **Dos corridas de FT: con y sin aug**, re-evaluadas *dentro* del fine-tuning; selección por **F1 de val** |
| 4 | Test de 300 sin intervalos de confianza; `mejor_global` se elegía por F1 de **test** | **Bootstrap 10 000 → IC 95 %**, evaluación adicional en `test_pool_rest` (752), y **selección por val** |
| 5 | Ítem 5 sin matriz de costos; el resumen decía "FN (sano→enfermo)" (invertido) | **Matriz de costos** + costo esperado + decisión por **mínimo riesgo** + sensibilidad enfermo-vs-sano; el error caro es **enfermo→sano** |
| 6 | Ítem 2 solo s/época y parámetros; "Params" de RF/XGB/SVM no eran comparables | **Latencia (ms/img), FLOPs y MB** del modelo; la complejidad de los clásicos va en **columna aparte** |
| 7 | LIME con 500 muestras / 10 imágenes, explicaba la clase *predicha*, texto de conclusión escrito de antemano | **1000 muestras × 15 imágenes**, explica **clase real y predicha**, **% de importancia dentro de la hoja** (máscara verde del EDA) + **Grad-CAM**; la conclusión se escribe **desde los números** |
| 8 | F1 por fuente con ~57 imágenes de 2019 era ruido | **Clasificador de `source` desde los embeddings** (¿el embedding captura el dominio?) |
| 9 | Ítem 8 sin robustez ni calibración | Curvas **F1 vs severidad** (blur, ruido, brillo, recorte), **ECE** + temperature scaling y **umbral de abstención**; las 33 "no-hojas" como **set OOD/rechazo** |
| 10 | SHA-1 solo detecta duplicados exactos | **Diagnóstico de casi-duplicados con pHash** (no re-particiona: se mantiene el split, por decisión) |
| 11 | Diccionario: CBB = "Brown Streak (bacteriosis/CFMV)", CGM = "Green Mosaic" | **Corregido**: CBB = *Bacterial Blight*; CBSD = *Brown Streak* (virus); CGM = *Green Mottle* |
| 12 | E1/E3 equivalían a una regresión logística sobre los embeddings (horas de GPU de más) | **Linear probe** sobre los mismos embeddings (barato, misma pregunta) |
| 13 | Faltaban ROC-AUC / PR-AUC; el EDA prometía brillo/contraste ±10 % | Ambos implementados (`ColorJitter` con ±10 % de brillo/contraste) |

**Alcance acordado:** entrena **solo MobileNetV2**; EfficientNet-B7 entra únicamente como **fuente de embeddings** (caché de la v1). Quedan fuera: ítem 7 (Optuna), factorial de backbones y curva de aprendizaje por fracción de datos.


## Fase 0 — Entorno, semillas y configuración


In [ ]:
import os, sys, json, time, math, random, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
from torchvision import transforms as T

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, average_precision_score,
                             log_loss, roc_curve, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from IPython.display import display, Markdown

try:
    import imagehash
except Exception:
    imagehash = None
try:
    import thop
except Exception:
    thop = None

SMOKE = os.environ.get("SMOKE", "0") == "1"
TAG = "_smoke" if SMOKE else ""

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
T_INICIO = time.time()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("torch:", torch.__version__, "| timm:", timm.__version__)
print("device:", DEVICE, "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("modo SMOKE:", SMOKE)
print("opcionales -> imagehash:", imagehash is not None, "| thop:", thop is not None)
print("Grad-CAM: implementacion propia con hooks (sin dependencia extra)")


In [ ]:
BASE = Path("D:/final_inteligencia")
DIR_IMG  = BASE / "datos" / "cambios 01 - no hojas" / "imagenes filtradas"
DIR_NOH  = BASE / "datos" / "cambios 01 - no hojas" / "outliers"
DIR_LABELS = BASE / "datos" / "cambios 01 - no hojas" / "labels_filtrado.csv"
MANIFEST   = BASE / "datos" / "splits" / "manifest_20pct.csv"
PHASH_CACHE = BASE / "datos" / "splits" / "phash_splits.csv"
DIR_MODEL = BASE / "modelos"          # checkpoints v1 (E1-E9): intactos
DIR_V2    = DIR_MODEL / "v2"          # artefactos de este notebook
DIR_REPORT  = BASE / "reportes"
DIR_INFORMES = DIR_REPORT / "informes_v2"
DIR_LIME  = DIR_REPORT / "lime_gradcam_v2"
DIR_ROB   = DIR_REPORT / "robustez_v2"
CSV_OUT   = DIR_REPORT / "03_resultados_v2.csv"
for _d in (DIR_V2, DIR_INFORMES, DIR_LIME, DIR_ROB):
    _d.mkdir(parents=True, exist_ok=True)

SEED = 42
SIZE = 384                      # resize directo, sin crop
CLASS_NAMES = ["cbb", "cbsd", "cgm", "cmd", "healthy"]
NOMBRE = {
    0: "CBB - Cassava Bacterial Blight (bacteriosis)",
    1: "CBSD - Cassava Brown Streak Disease (virus CFMV)",
    2: "CGM - Cassava Green Mottle (moteado verde)",
    3: "CMD - Cassava Mosaic Disease (mosaico)",
    4: "Saludable (sin sintomas)",
}
NUM_CLASSES = 5
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

FT_CFG = dict(
    epochs=1 if SMOKE else 20,
    patience=1 if SMOKE else 5,
    warmup=0 if SMOKE else 3,
    lr_bb=1e-4, lr_head=1e-3, wd=0.01,
    label_smoothing=0.1,
    batch=8 if SMOKE else 32,
    dropout=0.3,
)
NUM_WORKERS = 0  # la clase del dataset vive en el notebook: spawn (Windows) no puede picklearla
LIME_SAMPLES = 60 if SMOKE else 1000
LIME_POR_CLASE = 1 if SMOKE else 3
BOOT_N = 200 if SMOKE else 10000
N_IMG_INFORME = 6 if SMOKE else 18
N_IMG_ERRORES = 6 if SMOKE else 12

SEV = dict(
    blur=[0.0, 0.5, 1.0, 2.0, 3.0],
    ruido=[0.0, 0.02, 0.05, 0.10, 0.20],
    brillo=[0.0, 10, 20, 30],
    recorte=[1.0, 0.9, 0.8, 0.7],
)
if SMOKE:
    SEV = {k: [v[0], v[-1]] for k, v in SEV.items()}

print(json.dumps(FT_CFG, indent=1))
print("SIZE", SIZE, "| workers", NUM_WORKERS, "| LIME", LIME_SAMPLES,
      "| bootstrap", BOOT_N, "| img informe", N_IMG_INFORME)


## Fase 1 — Diccionario corregido, split reutilizado y casi-duplicados

- **Split**: se **lee** `manifest_20pct.csv` (idéntico al de la v1). Estratificado por **clase × source**, semilla 42, `image_id` jamás como feature.
- **Diccionario corregido**: la v1/EDA describían CBB como *"Brown Streak (bacteriosis/CFMV)"* — eso es la descripción de **CBSD**.
- **Casi-duplicados**: SHA-1 no detecta varias tomas de la misma planta; se mide con **pHash** (hamming ≤ 6) y se **reporta** (no se re-particiona).


In [ ]:
print("=== Diccionario de clases (CORREGIDO) ===")
display(pd.DataFrame({"codigo": range(5),
                      "clase": CLASS_NAMES,
                      "descripcion": [NOMBRE[i] for i in range(5)]}))
print("Correccion respecto de la v1/EDA: CBB = Bacterial Blight (bacteriosis), NO 'Brown Streak';",
      "\nCBSD = Brown Streak (virus CFMV); CGM = Green Mottle, NO 'Green Mosaic'.")

df_lab = pd.read_csv(DIR_LABELS)
assert len(df_lab) == 26304 and df_lab["image_id"].is_unique

manifest = pd.read_csv(MANIFEST)
assert manifest["image_id"].is_unique
assert set(manifest["split"]) == {"train", "val", "test", "test_pool_rest", "out_subset"}

chk = manifest.merge(df_lab, on="image_id", how="left", suffixes=("", "_lab"))
assert len(chk) == len(manifest) and chk["label_lab"].notna().all()
assert (chk["label"] == chk["label_lab"]).all(), "manifest y labels_filtrado.csv no coinciden"
assert chk["label_name"].eq(chk["label"].map(dict(enumerate(CLASS_NAMES)))).all()

print("manifest LEIDO (no regenerado):", MANIFEST)
print("filas:", len(manifest), "| subset 20 % =", int((manifest.split != "out_subset").sum()))


In [ ]:
if SMOKE:
    keep = set()
    for sp, n in [("train", 8), ("val", 4), ("test", 4)]:
        keep |= set(manifest[manifest.split == sp].groupby("label", group_keys=False).head(n).index)
    mask_reduc = (~manifest.index.isin(keep)) & manifest["split"].isin(["train", "val", "test"])
    manifest.loc[mask_reduc, "split"] = "out_subset"
    print("SMOKE: train/val/test reducidos a 8/4/4 por clase (el manifest en disco NO se toca)")
    display(manifest[manifest.split != "out_subset"].groupby(["split", "label"]).size().unstack(0))

df_test = manifest[manifest.split == "test"].reset_index(drop=True)
df_val = manifest[manifest.split == "val"].reset_index(drop=True)

print("\n== conteo por split ==")
print(manifest["split"].value_counts().to_string())
print("\n== clase x split ==")
display(pd.crosstab(manifest["label"].map(NOMBRE).str.split(" - ").str[0],
                    manifest["split"])[["train", "val", "test", "test_pool_rest", "out_subset"]])
print("\n== fuente x split ==")
display(pd.crosstab(manifest["source"], manifest["split"]))

s_tr = set(manifest[manifest.split == "train"]["image_id"])
s_va = set(manifest[manifest.split == "val"]["image_id"])
s_te = set(manifest[manifest.split == "test"]["image_id"])
s_tpr = set(manifest[manifest.split == "test_pool_rest"]["image_id"])
assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te)
assert s_tr.isdisjoint(s_tpr) and s_va.isdisjoint(s_tpr) and s_te.isdisjoint(s_tpr)
tc = manifest[manifest.split == "test"]["label"].value_counts().sort_index()
assert (tc == (4 if SMOKE else 60)).all()
print("\nOK: sin fuga entre train/val/test/test_pool_rest | test balanceado =", int(tc.sum()))

orden = ["train", "val", "test", "test_pool_rest", "out_subset"]
ct = manifest[manifest.split != "out_subset"].groupby(["label_name", "split"]).size().unstack(0)[CLASS_NAMES]
ax = ct.reindex(orden[:4]).plot(kind="bar", figsize=(10, 4), rot=0)
ax.set_title("Distribucion de clases por split (subset 20 %, identico a la v1)")
ax.set_ylabel("imagenes"); ax.set_xlabel("split")
plt.tight_layout(); plt.show()


### 1.1 Casi-duplicados (pHash) — diagnostico, sin re-particionar

Varias fotos de campo son tomas de la misma planta: eso **infla el test** si caen en splits
distintos. SHA-1 no lo ve (no son bytes idénticos), pHash sí. **Este notebook NO cambia el
split** (decisión de mantener la comparabilidad con la v1); si hay pares cruzados, queda
documentado como limitación.


In [ ]:
if imagehash is None:
    print("OMITIDO: imagehash no disponible.")
else:
    sub = manifest[manifest.split != "out_subset"].copy()
    cache = (pd.read_csv(PHASH_CACHE, dtype=str) if PHASH_CACHE.exists()
             else pd.DataFrame(columns=["image_id", "phash"]))
    sub = sub.merge(cache.drop_duplicates("image_id"), on="image_id", how="left")
    falta = sub[sub["phash"].isna()]
    if len(falta):
        print(f"calculando pHash de {len(falta)} imagenes...")
        t0 = time.time()
        nuevos = {}
        for i, f in enumerate(falta["image_id"], 1):
            try:
                nuevos[f] = str(imagehash.phash(Image.open(DIR_IMG / f).convert("L")))
            except Exception:
                nuevos[f] = None
            if i % 500 == 0:
                print(f"  {i}/{len(falta)} ({time.time() - t0:.0f}s)")
        sub["phash"] = sub["phash"].fillna(sub["image_id"].map(nuevos))
        sub[["image_id", "phash"]].dropna().to_csv(PHASH_CACHE, index=False)
        print(f"cache guardado en {PHASH_CACHE.name}")

    ord_ = sub.dropna(subset=["phash"]).reset_index(drop=True)
    arr = np.array([int(h, 16) for h in ord_["phash"]], dtype=np.uint64)
    a = np.frombuffer(arr.tobytes(), dtype=np.uint8).reshape(len(arr), 8)
    POPC = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
    sp = ord_["split"].values
    pi_l, pj_l, pd_l = [], [], []
    for i0 in range(0, len(a), 512):
        blk = a[i0:i0 + 512]
        dist = POPC[blk[:, None, :] ^ a[None, :, :]].sum(axis=2)
        ii, jj = np.nonzero(dist <= 6)
        ii = ii + i0
        keep = ii < jj
        pi, pj = ii[keep], jj[keep]
        pi_l.append(pi); pj_l.append(pj); pd_l.append(dist[pi - i0, pj])
    pi = np.concatenate(pi_l) if pi_l else np.array([], dtype=int)
    pj = np.concatenate(pj_l) if pj_l else np.array([], dtype=int)
    pd_ = np.concatenate(pd_l) if pd_l else np.array([], dtype=int)
    cruz = pi[sp[pi] != sp[pj]]
    print(f"imagenes con pHash: {len(ord_)}")
    print(f"pares con hamming <= 6 (casi-duplicados): {len(pi)}")
    print(f"  de los cuales entre SPLITS DISTINTOS: {len(cruz)}")
    if len(cruz):
        tab = {}
        for i in cruz:
            k = " <-> ".join(sorted([sp[i], sp[pj[list(pi).index(i)]]]))
            tab[k] = tab.get(k, 0) + 1
        display(pd.Series(tab, name="pares_cruzados").to_frame())
        print("LIMITACION DOCUMENTADA: si estos pares cruzan train/test, el test deja de ser")
        print("independiente. Corregirlo exige re-particionar (split por grupos), lo que anularia")
        print("los resultados de la v1 -> no se hace en esta corrida.")
    else:
        print("OK: ningun casi-duplicado cruza splits distintos.")


## Fase 2 — Infraestructura v2

**Preprocesamiento (corrección del ítem "pierde información diagnóstica"):**
- `Resize((384,384))` **directo**, sin `CenterCrop` ni `RandomCrop`: no se pierde el ancho de la imagen y no se corre la lesión fuera del cuadro. La deformación 4:3 → 1:1 es leve y una CNN con GAP acepta cualquier tamaño.
- **Aug solo en train**: flips + rotación ±15° + traslación 8 % + `ColorJitter(brillo/contraste ±10 %` — lo que el EDA §6 prometía y la v1 no tenía`)` + blur leve + ruido gaussiano leve.
- val/test: `Resize + Normalize` (determinístico).

**Entrenamiento:** fine-tuning **completo** (todo el backbone entrenable), AdamW con **lr diferencial** (backbone 1e-4, cabeza 1e-3), **warmup 3 + cosine**, `CrossEntropyLoss` con pesos de clase (solo train) + **label smoothing 0.1**, AMP + `channels_last`, early stopping por **macro-F1 de val**.


In [ ]:
class RuidoGaussiano:
    """Ruido aditivo sobre el tensor ya normalizado a [0,1] (antes de Normalize)."""
    def __init__(self, p=0.15, sigma=0.01):
        self.p, self.sigma = p, sigma
    def __call__(self, t):
        if float(torch.rand(1)) < self.p:
            t = (t + torch.randn_like(t) * self.sigma).clamp(0.0, 1.0)
        return t

def build_transforms(size=SIZE, aug=False):
    ops = [T.Resize((size, size), interpolation=T.InterpolationMode.BILINEAR)]
    if aug:
        ops += [
            T.RandomHorizontalFlip(0.5),
            T.RandomVerticalFlip(0.5),
            T.RandomRotation(15),
            T.RandomAffine(0, translate=(0.08, 0.08), scale=(0.95, 1.05)),
            T.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05, hue=0.02),
            T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.2),
        ]
    ops += [T.ToTensor()]
    if aug:
        ops += [RuidoGaussiano(p=0.15, sigma=0.01)]
    ops += [T.Normalize(MEAN, STD)]
    return T.Compose(ops)

tf_train = build_transforms(SIZE, True)
tf_eval = build_transforms(SIZE, False)
_demo = Image.open(DIR_IMG / df_test.image_id.iloc[0]).convert("RGB")
print("transforms listas")
print("origen:", _demo.size, "-> train aug:", tuple(tf_train(_demo).shape),
      "| eval:", tuple(tf_eval(_demo).shape))


In [ ]:
class MandiocaDataset(Dataset):
    def __init__(self, mdf, split, transform):
        self.df = mdf[mdf["split"] == split].reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(DIR_IMG / r["image_id"]).convert("RGB")
        return self.transform(img), int(r["label"])

class Clf(nn.Module):
    def __init__(self, backbone_name="mobilenetv2_100", dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(self.backbone.num_features, NUM_CLASSES))
    def forward(self, x):
        return self.head(self.backbone(x))
    def featurizador(self, x):
        return self.backbone(x)

def make_loader(ds, batch, shuffle=False, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle,
                      num_workers=NUM_WORKERS,
                      pin_memory=(DEVICE.type == "cuda"),
                      generator=g,
                      persistent_workers=(NUM_WORKERS > 0),
                      drop_last=False)

tr_labels = manifest.loc[manifest.split == "train", "label"].values
pesos = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=tr_labels)
CLASS_W = torch.tensor(pesos, dtype=torch.float32, device=DEVICE)
print("pesos de clase (calculados SOLO sobre train):",
      {CLASS_NAMES[i]: round(float(p), 3) for i, p in enumerate(pesos)})


In [ ]:
def armar_metricas(y_true, y_pred, y_prob=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    per = f1_score(y_true, y_pred, average=None, labels=np.arange(NUM_CLASSES), zero_division=0)
    out = dict(f1_macro=float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
               f1_per_class=[float(v) for v in per],
               balanced_acc=float(balanced_accuracy_score(y_true, y_pred)),
               y_true=y_true.tolist(), y_pred=y_pred.tolist())
    if y_prob is not None:
        y_prob = np.asarray(y_prob)
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_prob, average="macro",
                                                 multi_class="ovr", labels=np.arange(NUM_CLASSES)))
            out["pr_auc"] = float(average_precision_score(y_true, y_prob, average="macro",
                                                          labels=np.arange(NUM_CLASSES)))
        except Exception:
            out["roc_auc"] = None; out["pr_auc"] = None
        out["y_prob"] = y_prob.round(6).tolist()
    return out

@torch.no_grad()
def evaluar_completo(model, ds, batch=64, transform=None):
    """Devuelve metricas + probabilidades (necesarias para costos, calibracion y AUC)."""
    model.eval()
    dl = make_loader(ds, batch, shuffle=False)
    probs, ys = [], []
    for x, y in dl:
        p = torch.softmax(model(x.to(DEVICE, non_blocking=True)), dim=1)
        probs.append(p.cpu().numpy()); ys.append(y.numpy())
    y_prob = np.concatenate(probs).astype(np.float64)
    y_true = np.concatenate(ys)
    return armar_metricas(y_true, y_prob.argmax(1), y_prob)

def cargar_modelo(backbone, ruta):
    m = Clf(backbone).to(DEVICE)
    m.load_state_dict(torch.load(ruta, map_location=DEVICE))
    m.eval()
    return m


In [ ]:
def _lr_lambda(ep, warmup, total):
    if warmup > 0 and ep < warmup:
        return float(ep + 1) / float(warmup)
    t = (ep - warmup) / max(1, total - warmup)
    t = min(max(t, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * t))

def entrenar_ft(exp_id, aug, backbone="mobilenetv2_100", cfg=None):
    """Fine-tuning COMPLETO del backbone + cabeza. Cache por checkpoint+json."""
    cfg = dict(FT_CFG) if cfg is None else dict(cfg)
    ckpt = DIR_V2 / (exp_id + TAG + ".pth")
    res_p = DIR_V2 / (exp_id + TAG + "_res.json")
    if ckpt.exists() and res_p.exists():
        print("[cache]", exp_id, "ya entrenado")
        return json.loads(res_p.read_text(encoding="utf-8"))
    batch = cfg["batch"]
    while True:
        try:
            return _ft_inner(exp_id, aug, backbone, cfg, batch, ckpt, res_p)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            batch = batch // 2
            if batch < 4:
                raise
            print("  [OOM] reintentando con batch =", batch, flush=True)

def _ft_inner(exp_id, aug, backbone, cfg, batch, ckpt, res_p):
    set_seed()
    ds_tr = MandiocaDataset(manifest, "train", build_transforms(SIZE, aug))
    ds_va = MandiocaDataset(manifest, "val", tf_eval)
    ds_te = MandiocaDataset(manifest, "test", tf_eval)
    dl_tr = make_loader(ds_tr, batch, shuffle=True)

    model = Clf(backbone, dropout=cfg["dropout"]).to(DEVICE)
    model = model.to(memory_format=torch.channels_last)

    opt = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": cfg["lr_bb"], "weight_decay": cfg["wd"]},
        {"params": model.head.parameters(), "lr": cfg["lr_head"], "weight_decay": 0.0},
    ], betas=(0.9, 0.999))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda ep: _lr_lambda(ep, cfg["warmup"], cfg["epochs"]))
    crit = nn.CrossEntropyLoss(weight=CLASS_W, label_smoothing=cfg["label_smoothing"])
    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    n_total = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{exp_id}: aug={aug} train={len(ds_tr)} val={len(ds_va)} test={len(ds_te)} "
          f"batch={batch} params={n_total:,} (entrenables={n_train:,})", flush=True)

    best_f1, best_state, wait, history, t0 = -1.0, None, 0, [], time.time()
    for ep in range(cfg["epochs"]):
        model.train()
        ep_loss, n_seen, ep_t = 0.0, 0, time.time()
        lr_now = opt.param_groups[0]["lr"]
        for x, y in dl_tr:
            x = x.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
            y = y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                loss = crit(model(x), y)
            if not torch.isfinite(loss):
                print(f"  [{exp_id}] loss no finita, batch descartado", flush=True)
                continue
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            ep_loss += float(loss) * len(y); n_seen += len(y)
        sched.step()
        ev = evaluar_completo(model, ds_va, batch=max(16, batch))
        history.append(dict(epoch=ep + 1,
                            train_loss=round(ep_loss / max(1, n_seen), 4),
                            val_f1=round(ev["f1_macro"], 4),
                            lr=round(lr_now, 9),
                            epoch_time_s=round(time.time() - ep_t, 1)))
        print(f"  ep{ep + 1}/{cfg['epochs']} loss={ep_loss / max(1, n_seen):.4f} "
              f"valF1={ev['f1_macro']:.4f} lr={lr_now:.2e} ({history[-1]['epoch_time_s']}s)",
              flush=True)
        if ev["f1_macro"] > best_f1 + 1e-4:
            best_f1, wait = ev["f1_macro"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= cfg["patience"]:
                print("  early stopping"); break

    torch.save(best_state, ckpt)
    model.load_state_dict(best_state)
    va = evaluar_completo(model, ds_va, batch=max(16, batch))
    te = evaluar_completo(model, ds_te, batch=max(16, batch))
    res = dict(id=exp_id,
               metodo=("FT MobileNetV2 completo (sin aug)" if not aug
                       else "FT MobileNetV2 completo (con aug)"),
               backbone=backbone, size=SIZE, aug=aug, epochs_run=len(history),
               params=n_total, params_entrenables=n_train,
               tiempo_total_s=round(time.time() - t0, 1),
               tiempo_ep_s=round(float(np.mean([h["epoch_time_s"] for h in history])), 1),
               cfg=cfg, val=va, test=te, history=history)
    res_p.write_text(json.dumps(res, indent=1), encoding="utf-8")
    print(f"{exp_id} listo: valF1={va['f1_macro']:.4f} testF1={te['f1_macro']:.4f} "
          f"({res['tiempo_total_s']}s)")
    return res

RESULTADOS = {}


## Fase 3 — Informe de desempeño por modelo: matriz de confusión + ejemplos + Grad-CAM

Funciones reutilizables que se llaman **después de entrenar/evaluar cada modelo** (igual que en
`ref personal\pruebas_marzo_Manihot_diseases_detection.ipynb`):

1. `classification_report` con los 5 nombres de clase;
2. **matriz de confusión con conteos** + **normalizada por fila**;
3. **grilla de imágenes de ejemplo** del test con `Pred:` / `Real:` (verde = bien, rojo = mal) y
   la confianza de la predicción;
4. si se pasa el modelo (CNN), una tira de **Grad-CAM** para las mismas imágenes.

Todo se guarda en `reportes\informes_v2\<modelo>.png`.


In [ ]:
def _guardar(slug):
    p = DIR_INFORMES / (slug + ".png")
    plt.savefig(p, dpi=110, bbox_inches="tight")
    plt.show()
    print("  ->", p)

def informe_visual(nombre, res, modelo=None, n_imgs=N_IMG_INFORME, inline=True, gradcam_n=0):
    """Reporte completo de un modelo: classification_report + CM + ejemplos (+ Grad-CAM)."""
    y_true = np.array(res["y_true"]); y_pred = np.array(res["y_pred"])
    y_prob = np.array(res["y_prob"]) if res.get("y_prob") is not None else None
    slug = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in nombre)

    print("=" * 78)
    print("MODELO:", nombre)
    print("=" * 78)
    print(classification_report(y_true, y_pred, labels=np.arange(NUM_CLASSES),
                                target_names=CLASS_NAMES, digits=3, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    with np.errstate(invalid="ignore", divide="ignore"):
        cm_n = cm / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[0].set_title(f"{nombre} - matriz de confusion (conteo)")
    sns.heatmap(cm_n, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=axes[1],
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    axes[1].set_title(f"{nombre} - normalizada por fila | macro-F1 = {res['f1_macro']:.4f}")
    for ax in axes:
        ax.set_xlabel("predicha"); ax.set_ylabel("real")
    plt.tight_layout(); _guardar(slug + "_cm")

    if n_imgs > 0:
        k = min(n_imgs, len(y_true))
        idx = np.random.RandomState(SEED).choice(len(y_true), size=k, replace=False)
        cols = 6 if k > 6 else max(1, k)
        rows = int(math.ceil(k / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(2.9 * cols, 3.3 * rows), squeeze=False)
        for j, i in enumerate(idx):
            ax = axes[j // cols][j % cols]
            ax.imshow(Image.open(DIR_IMG / df_test.image_id.iloc[int(i)]))
            ok = y_pred[i] == y_true[i]
            conf = "" if y_prob is None else f"\n{100 * y_prob[i].max():.0f}%"
            ax.set_title(f"Pred: {CLASS_NAMES[y_pred[i]]}\nReal: {CLASS_NAMES[y_true[i]]}{conf}",
                         color="green" if ok else "red", fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
        for j in range(k, rows * cols):
            axes[j // cols][j % cols].axis("off")
        fig.suptitle(f"{nombre} - ejemplos del test (verde = acierto, rojo = error)", fontsize=12)
        plt.tight_layout(); _guardar(slug + "_ejemplos")

    if modelo is not None and gradcam_n > 0:
        muestra = np.random.RandomState(SEED + 1).choice(len(y_true),
                                                         size=min(gradcam_n, len(y_true)),
                                                         replace=False)
        gradcam_tira(modelo, [df_test.image_id.iloc[int(i)] for i in muestra],
                     [int(y_true[i]) for i in muestra],
                     [int(y_pred[i]) for i in muestra],
                     titulo=nombre, slug=slug + "_gradcam")
    if not inline:
        plt.close("all")
    return confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))


In [ ]:
def ultima_capa_conv(model):
    last = None
    for m in model.backbone.modules():
        if isinstance(m, nn.Conv2d):
            last = m
    return last

def gradcam(model, x, clase=None):
    """Grad-CAM propio con hooks. x: [1,3,H,W] normalizado. Devuelve heatmap HxW en [0,1]."""
    capa = ultima_capa_conv(model)
    feats, grads = {}, {}
    h1 = capa.register_forward_hook(lambda m, i, o: feats.__setitem__("v", o))
    h2 = capa.register_full_backward_hook(lambda m, gi, go: grads.__setitem__("v", go[0]))
    try:
        model.eval()
        x = x.detach().clone().requires_grad_(True)
        logits = model(x)
        score = logits[0, clase] if clase is not None else logits[0].max()
        model.zero_grad(set_to_none=True)
        score.backward()
        f, g = feats["v"], grads["v"]
        w = g.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((w * f).sum(dim=1, keepdim=True))
        cam = nn.functional.interpolate(cam, size=x.shape[-2:], mode="bilinear",
                                        align_corners=False)[0, 0]
        cam = cam.detach().float().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    finally:
        h1.remove(); h2.remove()

def _tf_gradcam(img_id):
    return tf_eval(Image.open(DIR_IMG / img_id).convert("RGB")).unsqueeze(0).to(DEVICE)

def gradcam_tira(model, img_ids, y_true, y_pred, titulo="", slug=None, n=None):
    n = n or len(img_ids)
    fig, axes = plt.subplots(2, n, figsize=(3.0 * n, 6.4), squeeze=False)
    for j in range(n):
        img = Image.open(DIR_IMG / img_ids[j]).convert("RGB")
        x = _tf_gradcam(img_ids[j])
        cam_real = gradcam(model, x, clase=int(y_true[j]))
        cam_pred = gradcam(model, x, clase=int(y_pred[j]))
        axes[0][j].imshow(img); axes[0][j].imshow(cam_real, cmap="jet", alpha=0.45)
        axes[0][j].set_title(f"clase REAL\n{CLASS_NAMES[y_true[j]]}", fontsize=8)
        axes[1][j].imshow(img); axes[1][j].imshow(cam_pred, cmap="jet", alpha=0.45)
        axes[1][j].set_title(f"clase PREDICHA\n{CLASS_NAMES[y_pred[j]]}", fontsize=8)
        for ax in (axes[0][j], axes[1][j]):
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"{titulo} - Grad-CAM (importancia por pixel)", fontsize=12)
    plt.tight_layout()
    if slug:
        _guardar(slug)
    else:
        plt.show()


In [ ]:
def informe_errores(nombre, res, n_imgs=N_IMG_ERRORES):
    """Muestra los errores del modelo: primero los 'enfermo -> sano' (los mas caros)."""
    y_true = np.array(res["y_true"]); y_pred = np.array(res["y_pred"])
    y_prob = np.array(res["y_prob"]) if res.get("y_prob") is not None else None
    err = np.where(y_true != y_pred)[0]
    if len(err) == 0:
        print(nombre, "-> sin errores en el test"); return
    caro = [i for i in err if y_true[i] != 4 and y_pred[i] == 4]
    otros = [i for i in err if not (y_true[i] != 4 and y_pred[i] == 4)]
    idx = (caro + otros)[:n_imgs]
    cols = 6 if len(idx) > 6 else max(1, len(idx))
    rows = int(math.ceil(len(idx) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.9 * cols, 3.3 * rows), squeeze=False)
    for j, i in enumerate(idx):
        ax = axes[j // cols][j % cols]
        ax.imshow(Image.open(DIR_IMG / df_test.image_id.iloc[int(i)]))
        extra = "" if y_prob is None else f"\nconf {100 * y_prob[i].max():.0f}%"
        tipo = "ENFERMO->SANO" if (y_true[i] != 4 and y_pred[i] == 4) else "error"
        ax.set_title(f"{tipo}\nPred: {CLASS_NAMES[y_pred[i]]} | Real: {CLASS_NAMES[y_true[i]]}{extra}",
                     color="red", fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
    for j in range(len(idx), rows * cols):
        axes[j // cols][j % cols].axis("off")
    fig.suptitle(f"{nombre} - errores (primero los mas caros: enfermo -> sano)", fontsize=12)
    plt.tight_layout(); _guardar("errores_" + "".join(c if c.isalnum() or c in "-_" else "_"
                                                      for c in nombre))


## Fase 4 — Embeddings, linear probe y clasificadores clásicos (ítem 4)

- **Embeddings**: GAP del backbone **ImageNet** a 384 px (MobileNetV2, nuevo cache en `modelos\v2\`)
  y los de la **v1** para EfficientNet-B7 @600 (caché reusado, sin costo).
- **Linear probe** = `LogisticRegression` sobre esos embeddings: es exactamente lo que medían
  E1/E3 en la v1 (backbone congelado + cabeza lineal), pero **sin pasar por la red en GPU**.
- Después se agregan los embeddings del **MobileNetV2 fine-tuneado** (Fase 5).
- Clásicos: RandomForest, SVM RBF (grid sobre **val**), XGBoost y un MLP sobre embeddings.


In [ ]:
def extraer_embeddings(model, size, split, cache_name, batch=None):
    """Embeddings GAP de un modelo, con cache. El orden es el de `manifest[split]`."""
    cache = DIR_V2 / (cache_name + "_" + split + TAG + ".npz")
    if cache.exists():
        d = np.load(cache)
        print(f"[cache] {cache_name} {split}: {d['X'].shape}")
        return d["X"], d["y"]
    model.eval()
    ds = MandiocaDataset(manifest, split, build_transforms(size, False))
    dl = make_loader(ds, batch or (64 if size < 384 else 32), shuffle=False)
    F, L = [], []
    with torch.no_grad():
        for x, y in dl:
            f = model.featurizador(x.to(DEVICE, non_blocking=True)).float().cpu().numpy()
            F.append(f); L.append(y.numpy())
    X = np.concatenate(F).astype(np.float32)
    yy = np.concatenate(L)
    np.savez_compressed(cache, X=X, y=yy)
    print(f"emb {cache_name} {split}: {X.shape}")
    return X, yy

EMB = {}

def _set(nombre, bb, size, split, model=None):
    if model is None:
        if size == 600:                      # reusa el cache de la v1 (B7 @600)
            viejo = DIR_MODEL / f"emb_{bb}_{split}{TAG}.npz"
            if viejo.exists():
                d = np.load(viejo)
                print(f"[v1] {bb} {split}: {d['X'].shape}")
                return d["X"].astype(np.float32), d["y"]
        model = Clf(bb).to(DEVICE)
    return extraer_embeddings(model, size, split, f"emb_{nombre}_{size}")

for bb, size, nombre in [("mobilenetv2_100", SIZE, "mv2"),
                         ("tf_efficientnet_b7.ra_in1k", 600, "b7")]:
    EMB[nombre] = {}
    for sp in ["train", "val", "test"]:
        EMB[nombre][sp] = _set(nombre, bb, size, sp)
    assert EMB[nombre]["train"][1].shape[0] == int((manifest.split == "train").sum())
    assert EMB[nombre]["val"][1].shape[0] == int((manifest.split == "val").sum())
    assert EMB[nombre]["test"][1].shape[0] == int((manifest.split == "test").sum())
print("embeddings listos:", {k: v["train"][0].shape for k, v in EMB.items()})


In [ ]:
def metricas_sk(y_true, y_pred, y_prob=None):
    return armar_metricas(y_true, y_pred, y_prob)

def registrar_ml(exp_id, metodo, backbone, extra, val_m, test_m, complejidad, t_s,
                 size=None, aug=None):
    res = dict(id=exp_id, metodo=metodo, backbone=backbone, size=size, aug=aug,
               epochs_run=None, params=complejidad, tiempo_total_s=round(t_s, 1),
               tiempo_ep_s=None, extra=extra, val=val_m, test=test_m)
    (DIR_V2 / (exp_id + TAG + "_res.json")).write_text(json.dumps(res, indent=1),
                                                       encoding="utf-8")
    return res

RESULTADOS_ML = {}

def linear_probe(nombre, emb):
    """Regresion logistica sobre embeddings = lo que median E1/E3 (barato, sin GPU)."""
    ck = DIR_V2 / (f"LIN_{nombre}" + TAG + "_res.json")
    if ck.exists():
        r = json.loads(ck.read_text(encoding="utf-8"))
        RESULTADOS_ML[r["id"]] = r
        print("[cache]", r["id"]); return r
    t0 = time.time()
    Xtr, ytr = emb["train"]; Xv, yv = emb["val"]; Xt, yt = emb["test"]
    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xv_s, Xt_s = sc.transform(Xtr), sc.transform(Xv), sc.transform(Xt)
    filas, mejor = [], None
    for C in [0.1, 1.0, 10.0]:
        clf = LogisticRegression(C=C, max_iter=3000, solver="lbfgs", random_state=SEED)
        clf.fit(Xtr_s, ytr)
        f1v = float(f1_score(yv, clf.predict(Xv_s), average="macro", zero_division=0))
        filas.append(dict(C=C, f1_val=round(f1v, 4)))
        if mejor is None or f1v > mejor[0]:
            mejor = (f1v, C, clf)
    _, C_best, clf = mejor
    print(f"  {nombre}: grid", filas, "-> C =", C_best)
    r = registrar_ml(f"LIN_{nombre}", "Linear probe (logistic reg. sobre embeddings)",
                     emb.get("_bb", ""),
                     f"scaler+logreg C={C_best} | " + str(filas),
                     metricas_sk(yv, clf.predict(Xv_s), clf.predict_proba(Xv_s)),
                     metricas_sk(yt, clf.predict(Xt_s), clf.predict_proba(Xt_s)),
                     int(np.sum(clf.coef_.shape)), time.time() - t0)
    RESULTADOS_ML[r["id"]] = r
    print(f"  {nombre} linear probe: val {r['val']['f1_macro']:.4f} | test {r['test']['f1_macro']:.4f}")
    return r

for k in EMB:
    EMB[k]["_bb"] = "mobilenetv2_100" if k == "mv2" else "tf_efficientnet_b7.ra_in1k"
for k in ["mv2", "b7"]:
    linear_probe(k, EMB[k])


In [ ]:
def pesos_balanceados(y):
    n = len(y); counts = np.bincount(np.asarray(y), minlength=NUM_CLASSES)
    return n / (NUM_CLASSES * counts[np.asarray(y)])

def clasicos_rf_svm(nombre, emb):
    bb = emb["_bb"]
    Xtr, ytr = emb["train"]; Xv, yv = emb["val"]; Xt, yt = emb["test"]

    fid = f"E6_{nombre}"
    ck = DIR_V2 / (fid + TAG + "_res.json")
    if ck.exists():
        r = json.loads(ck.read_text(encoding="utf-8")); RESULTADOS_ML[r["id"]] = r
        print("[cache]", r["id"])
    else:
        t0 = time.time()
        clf = RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample",
                                     random_state=SEED, n_jobs=-1)
        clf.fit(Xtr, ytr)
        r = registrar_ml(fid, "RandomForest", bb, "n=500",
                         metricas_sk(yv, clf.predict(Xv), clf.predict_proba(Xv)),
                         metricas_sk(yt, clf.predict(Xt), clf.predict_proba(Xt)),
                         500, time.time() - t0)
        RESULTADOS_ML[r["id"]] = r
        print(f"  {fid} test F1 = {r['test']['f1_macro']:.4f}")

    fid = f"E7_{nombre}"
    ck = DIR_V2 / (fid + TAG + "_res.json")
    if ck.exists():
        r = json.loads(ck.read_text(encoding="utf-8")); RESULTADOS_ML[r["id"]] = r
        print("[cache]", r["id"])
    else:
        t0 = time.time()
        sc = StandardScaler().fit(Xtr)
        Xtr_s, Xv_s, Xt_s = sc.transform(Xtr), sc.transform(Xv), sc.transform(Xt)
        mejor = None; grid = []
        for C in [1, 10]:
            for gamma in ["scale", 1e-3]:
                clf = SVC(kernel="rbf", C=C, gamma=gamma, class_weight="balanced",
                          probability=True, random_state=SEED)
                clf.fit(Xtr_s, ytr)
                f1v = float(f1_score(yv, clf.predict(Xv_s), average="macro", zero_division=0))
                grid.append(f"C={C},g={gamma}:{f1v:.4f}")
                if mejor is None or f1v > mejor[0]:
                    mejor = (f1v, C, gamma, clf)
        _, C_b, g_b, clf = mejor
        print(f"  {fid} grid {grid} -> C={C_b} gamma={g_b}")
        r = registrar_ml(fid, "SVM RBF", bb, f"C={C_b}, gamma={g_b} | " + "; ".join(grid),
                         metricas_sk(yv, clf.predict(Xv_s), clf.predict_proba(Xv_s)),
                         metricas_sk(yt, clf.predict(Xt_s), clf.predict_proba(Xt_s)),
                         int(len(clf.support_)), time.time() - t0)
        RESULTADOS_ML[r["id"]] = r
        print(f"  {fid} test F1 = {r['test']['f1_macro']:.4f}")

for k in ["mv2", "b7"]:
    clasicos_rf_svm(k, EMB[k])


In [ ]:
import xgboost as xgb

def entrenar_mlp(Xtr, ytr, Xv, yv, Xt, yt, epochs=150, batch=256):
    set_seed()
    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xv_s, Xt_s = sc.transform(Xtr), sc.transform(Xv), sc.transform(Xt)
    F = Xtr_s.shape[1]
    model = nn.Sequential(nn.Linear(F, 256), nn.ReLU(), nn.Dropout(0.3),
                          nn.Linear(256, NUM_CLASSES)).to(DEVICE)
    counts = np.bincount(np.asarray(ytr), minlength=NUM_CLASSES).astype(np.float64)
    w = torch.tensor(len(ytr) / (NUM_CLASSES * counts), dtype=torch.float32, device=DEVICE)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.long)
    Xv_t = torch.tensor(Xv_s, dtype=torch.float32, device=DEVICE)
    yv_t = torch.tensor(yv, dtype=torch.long, device=DEVICE)
    n = len(Xtr_t)
    best_f1, best_state, wait, t0 = -1.0, None, 0, time.time()
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            xb, yb = Xtr_t[idx].to(DEVICE), ytr_t[idx].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            f1v = f1_score(yv_t.cpu().numpy(), model(Xv_t).argmax(1).cpu().numpy(),
                           average="macro", zero_division=0)
        if f1v > best_f1 + 1e-4:
            best_f1, wait = f1v, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= 15:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        pv = model(Xv_t).softmax(1).cpu().numpy()
        pt = model(torch.tensor(Xt_s, dtype=torch.float32, device=DEVICE)).softmax(1).cpu().numpy()
    return pv, pt, model, time.time() - t0

def clasicos_xgb_mlp(nombre, emb):
    bb = emb["_bb"]
    Xtr, ytr = emb["train"]; Xv, yv = emb["val"]; Xt, yt = emb["test"]

    fid = f"E8_{nombre}"
    ck = DIR_V2 / (fid + TAG + "_res.json")
    if ck.exists():
        r = json.loads(ck.read_text(encoding="utf-8")); RESULTADOS_ML[r["id"]] = r
        print("[cache]", r["id"])
    else:
        t0 = time.time()
        clf = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                objective="multi:softprob", num_class=NUM_CLASSES,
                                eval_metric="mlogloss", tree_method="hist",
                                random_state=SEED, n_jobs=-1, reg_lambda=1.0)
        clf.fit(Xtr, ytr, sample_weight=pesos_balanceados(ytr),
                eval_set=[(Xv, yv)], verbose=False)
        r = registrar_ml(fid, "XGBoost", bb, "n=300, d=6, lr=0.1",
                         metricas_sk(yv, clf.predict(Xv), clf.predict_proba(Xv)),
                         metricas_sk(yt, clf.predict(Xt), clf.predict_proba(Xt)),
                         300, time.time() - t0)
        RESULTADOS_ML[r["id"]] = r
        print(f"  {fid} test F1 = {r['test']['f1_macro']:.4f}")

    fid = f"E9_{nombre}"
    ck = DIR_V2 / (fid + TAG + "_res.json")
    if ck.exists():
        r = json.loads(ck.read_text(encoding="utf-8")); RESULTADOS_ML[r["id"]] = r
        print("[cache]", r["id"])
    else:
        pv, pt, model_mlp, t_s = entrenar_mlp(Xtr, ytr, Xv, yv, Xt, yt,
                                              epochs=2 if SMOKE else 150)
        n_par = sum(p.numel() for p in model_mlp.parameters())
        r = registrar_ml(fid, "MLP (red sobre embeddings)", bb, "256 ocultas",
                         metricas_sk(yv, pv.argmax(1), pv),
                         metricas_sk(yt, pt.argmax(1), pt), n_par, t_s)
        RESULTADOS_ML[r["id"]] = r
        print(f"  {fid} test F1 = {r['test']['f1_macro']:.4f}")

for k in ["mv2", "b7"]:
    clasicos_xgb_mlp(k, EMB[k])


## Fase 5 — Fine-tuning completo de MobileNetV2: **con y sin augmentation**

Dos corridas con la misma receta (AdamW, lr diferencial 1e-4/1e-3, warmup 3 + cosine,
label smoothing 0.1, early stopping por macro-F1 de val). La **selección es por F1 de val**;
el test se reporta para ambas porque es una comparación experimental legítima (ablation).

Punto de referencia "antes" = **E1 de la v1** (backbone congelado, mismo split) — se lee de
`modelos\E1_res.json` sin re-entrenar.


In [ ]:
RESULTADOS["FT_noaug"] = entrenar_ft("FT_noaug", aug=False)


In [ ]:
RESULTADOS["FT_aug"] = entrenar_ft("FT_aug", aug=True)


In [ ]:
ruta_antes = DIR_MODEL / ("E1" + TAG + "_res.json")
res_antes = json.loads(ruta_antes.read_text(encoding="utf-8")) if ruta_antes.exists() else None

filas = []
if res_antes is not None:
    filas.append(dict(ID="E1 (v1)", Metodo="backbone congelado, cabeza lineal @224",
                      Aug="no", Epocas=res_antes["epochs_run"],
                      F1_val=res_antes["val"]["f1_macro"], F1_test=res_antes["test"]["f1_macro"],
                      ROC_AUC=res_antes["test"].get("roc_auc"),
                      Params_entrenables=2230277, s_epoc=res_antes["tiempo_ep_s"]))
for eid in ["FT_noaug", "FT_aug"]:
    r = RESULTADOS[eid]
    filas.append(dict(ID=eid, Metodo=r["metodo"], Aug="si" if r["aug"] else "no",
                      Epocas=r["epochs_run"], F1_val=r["val"]["f1_macro"],
                      F1_test=r["test"]["f1_macro"], ROC_AUC=r["test"].get("roc_auc"),
                      Params_entrenables=r["params_entrenables"], s_epoc=r["tiempo_ep_s"]))
tabla_ab = pd.DataFrame(filas)
display(tabla_ab.round(4))

mejor_ft = max(["FT_noaug", "FT_aug"], key=lambda e: RESULTADOS[e]["val"]["f1_macro"])
print(">>> Mejor FT por F1 de VAL:", mejor_ft,
      "| val", round(RESULTADOS[mejor_ft]["val"]["f1_macro"], 4),
      "| test (reportado una sola vez, no se usa para elegir):",
      round(RESULTADOS[mejor_ft]["test"]["f1_macro"], 4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for eid, r in RESULTADOS.items():
    h = pd.DataFrame(r["history"])
    axes[0].plot(h["epoch"], h["train_loss"], marker="o", label=eid)
    axes[1].plot(h["epoch"], h["val_f1"], marker="o", label=eid)
axes[0].set_title("perdida de train (FT)"); axes[0].set_xlabel("epoca"); axes[0].legend()
axes[1].set_title("macro-F1 de validacion (FT)"); axes[1].set_xlabel("epoca")
axes[1].set_ylim(0, 1); axes[1].legend()
plt.tight_layout(); plt.show()


In [ ]:
MODELOS = {}
for eid in ["FT_noaug", "FT_aug"]:
    r = RESULTADOS[eid]
    MODELOS[eid] = cargar_modelo(r["backbone"], DIR_V2 / (eid + TAG + ".pth"))
    informe_visual(eid, r["test"], modelo=MODELOS[eid], gradcam_n=(0 if SMOKE else 4))
    informe_errores(eid, r["test"])


## Fase 6 — Ítem 5: matriz de costos, costo esperado y mínimo riesgo

La consigna pide considerar **la importancia diferencial de los errores**. Definimos el costo
de la decisión `pred = j` cuando la clase real es `i` (fila = real, columna = predicha):

| real \ predicha | cbb | cbsd | cgm | cmd | **saludable** |
|---|---|---|---|---|---|
| **cbb / cbsd / cgm / cmd** (enfermo) | 0 | 2 | 2 | 2 | **5** |
| **saludable** (sano) | 1 | 1 | 1 | 1 | 0 |

- **Enfermo → sano = 5** (falso negativo: se despacha una planta enferma como sana → se propaga).
- **Sano → enfermo = 1** (falso positivo: costea una revisión innecesaria).
- **Entre enfermedades = 2** (se trata mal, pero al menos se trata).

> ⚠️ **Corrección de redacción:** en el resumen de la v1 decía *"FN (sano→enfermo)"*, lo cual está
> invertido. El **FN** (falso negativo) es **enfermo → sano**, y es justo el error caro.

Se reporta (a) el **costo realizado** promedio sobre el test, (b) el **costo esperado** que el
modelo pronuncia con sus probabilidades y (c) la decisión por **mínimo riesgo**
(`argmin_a  Σ_i C(i,a)·p(i|x)`), que a veces difiere del `argmax`.


In [ ]:
COSTO = np.array([
    [0, 2, 2, 2, 5],
    [2, 0, 2, 2, 5],
    [2, 2, 0, 2, 5],
    [2, 2, 2, 0, 5],
    [1, 1, 1, 1, 0],
], dtype=float)
display(pd.DataFrame(COSTO, index=CLASS_NAMES, columns=CLASS_NAMES)
        .rename_axis("real \\ predicha"))

def costo_realizado(y_true, y_pred):
    return float(np.mean(COSTO[np.asarray(y_true), np.asarray(y_pred)]))

def riesgo_accion(y_prob):
    """Riesgo esperado de cada accion a: R(a) = sum_i C(i,a) p(i|x) -> [N,5]."""
    return y_prob @ COSTO

def prediccion_min_riesgo(y_prob):
    return np.argmin(riesgo_accion(y_prob), axis=1)

def reporte_costos(nombre, res):
    m = res["test"] if ("test" in res and "y_true" in res["test"]) else res
    y_true = np.array(m["y_true"]); y_pred = np.array(m["y_pred"])
    y_prob = m.get("y_prob")
    fila = dict(ID=nombre,
                F1_test=m["f1_macro"],
                costo_realizado_argmax=costo_realizado(y_true, y_pred))
    if y_prob is not None:
        y_prob = np.asarray(y_prob)
        R = riesgo_accion(y_prob)
        a_mr = R.argmin(axis=1)
        fila["costo_esperado_argmax"] = float(np.mean(R[np.arange(len(y_true)), y_pred]))
        fila["costo_realizado_min_riesgo"] = costo_realizado(y_true, a_mr)
        fila["costo_esperado_min_riesgo"] = float(np.mean(R[np.arange(len(y_true)), a_mr]))
        fila["cambia_decisiones"] = int((a_mr != y_pred).sum())
    return fila

todos = {k: v for k, v in RESULTADOS.items()}
if res_antes is not None:
    todos["E1 (v1)"] = res_antes
for k, v in RESULTADOS_ML.items():
    todos[k] = v

tabla_costos = pd.DataFrame([reporte_costos(k, v) for k, v in todos.items()])
display(tabla_costos.round(4))
print("Costo menor = mejor decision bajo esta utilidad. Las columnas *_argmax usan la")
print("prediccion clasica (max probabilidad); *_min_riesgo eligen la accion de menor riesgo.")


In [ ]:
def sensibilidad_enfermo_vs_sano(y_true, y_prob):
    """Umbral sobre p(enfermo) = 1 - p(saludable)."""
    real_enfermo = np.asarray(y_true) != 4
    score = 1.0 - y_prob[:, 4]
    filas = []
    for tau in np.arange(0.0, 1.001, 0.05):
        pred_enfermo = score >= tau
        tp = int((pred_enfermo & real_enfermo).sum())
        fn = int((~pred_enfermo & real_enfermo).sum())
        tn = int((~pred_enfermo & ~real_enfermo).sum())
        fp = int((pred_enfermo & ~real_enfermo).sum())
        sens = tp / max(1, tp + fn)
        spec = tn / max(1, tn + fp)
        y_pred = np.where(pred_enfermo, y_prob[:, :4].argmax(1), 4)
        filas.append(dict(tau=round(float(tau), 2), sensibilidad=round(sens, 4),
                          especificidad=round(spec, 4),
                          costo=round(costo_realizado(y_true, y_pred), 4)))
    return pd.DataFrame(filas)

nombre_top = mejor_ft
y_true = np.array(RESULTADOS[nombre_top]["test"]["y_true"])
y_prob = np.array(RESULTADOS[nombre_top]["test"]["y_prob"])
curva = sensibilidad_enfermo_vs_sano(y_true, y_prob)
display(curva)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].plot(curva["especificidad"], curva["sensibilidad"], marker="o")
for _, r in curva.iloc[::3].iterrows():
    axes[0].annotate(f"t={r['tau']:.2f}", (r["especificidad"], r["sensibilidad"]), fontsize=7)
axes[0].set_xlabel("especificidad (sano bien detectado)")
axes[0].set_ylabel("sensibilidad (enfermo bien detectado)")
axes[0].set_title(f"{nombre_top} - curva sensibilidad vs especificidad (umbral p(enfermo))")
axes[1].plot(curva["tau"], curva["costo"], marker="o", color="tab:red")
axes[1].axvline(0.5, ls=":", color="gray", label="umbral 0.5 (argmax)")
axes[1].set_xlabel("umbral tau sobre p(enfermo)")
axes[1].set_ylabel("costo promedio realizado")
axes[1].set_title("costo vs umbral de decision")
axes[1].legend()
plt.tight_layout(); plt.show()

opt = curva.loc[curva["costo"].idxmin()]
print(f"Umbral de minimo costo en el test: tau = {opt['tau']:.2f} "
      f"(sensibilidad {opt['sensibilidad']:.3f}, especificidad {opt['especificidad']:.3f}, "
      f"costo {opt['costo']:.3f})")
print("NB: el umbral se deberia fijar sobre VAL y reportarse una vez sobre TEST; aca se muestra")
print("la curva completa del test solo como diagnostico de sensibilidad.")


## Fase 7 — Ítem 2: costo computacional (latencia, FLOPs, tamaño)

El ítem 2 pide el balance **capacidad vs costo** en un contexto móvil. Además de macro-F1:
**ms/imagen**, **FLOPs**, **MB del checkpoint** y parámetros. La complejidad de los clásicos
(`n_estimators`, vectores soporte) va en una **columna aparte**: no es comparable con parámetros
de una red.


In [ ]:
def medir_latencia(model, size=SIZE, batch=1, n_iter=200, warmup=20):
    if DEVICE.type != "cuda":
        return None
    model.eval()
    x = torch.randn(batch, 3, size, size, device=DEVICE).to(memory_format=torch.channels_last)
    with torch.no_grad():
        for _ in range(warmup):
            model(x)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_iter):
            model(x)
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_iter * 1000.0

def medir_flops(model, size=SIZE):
    if thop is None:
        return None
    model.eval()
    x = torch.randn(1, 3, size, size, device=DEVICE).to(memory_format=torch.channels_last)
    try:
        macs, _ = thop.profile(model, inputs=(x,), verbose=False)
        return float(macs)
    except Exception:
        return None

def filas_costo(nombre, metodo, params_total=None, params_train=None, complejidad=None,
                ckpt=None, modelo=None, size=SIZE):
    d = dict(ID=nombre, Metodo=metodo, Input=size,
             Params_totales=params_total, Params_entrenables=params_train,
             Complejidad_modelo_clasico=complejidad)
    d["MB_ckpt"] = round(ckpt.stat().st_size / 1e6, 2) if (ckpt and ckpt.exists()) else None
    if modelo is not None:
        d["ms_img_batch1"] = round(medir_latencia(modelo, size, 1), 3)
        d["ms_img_batch32"] = round(medir_latencia(modelo, size, 32, n_iter=50), 3)
        g = medir_flops(modelo, size)
        d["GFLOPs"] = None if g is None else round(g / 1e9, 3)
    return d

filas_c = []
for eid in ["FT_noaug", "FT_aug"]:
    r = RESULTADOS[eid]
    filas_c.append(filas_costo(eid, r["metodo"], r["params"], r["params_entrenables"],
                               ckpt=DIR_V2 / (eid + TAG + ".pth"), modelo=MODELOS[eid]))
if res_antes is not None:
    filas_c.append(filas_costo("E1 (v1)", "backbone congelado (cabeza lineal)",
                               2230277, 6405, ckpt=DIR_MODEL / ("E1" + TAG + ".pth"),
                               modelo=None, size=224))
tabla_costo = pd.DataFrame(filas_c)
display(tabla_costo)

print("\n-- complejidad de los clasificadores clasicos (NO comparable con params de red) --")
comp = []
for k, v in RESULTADOS_ML.items():
    comp.append(dict(ID=k, Metodo=v["metodo"], complejidad_reportada=v["params"],
                     nota=("n_estimators" if v["metodo"].startswith("RandomForest") or
                           v["metodo"].startswith("XGBoost") else
                           "vectores soporte" if v["metodo"].startswith("SVM") else
                           "coeficientes logreg" if v["metodo"].startswith("Linear") else
                           "params de la MLP")))
display(pd.DataFrame(comp))


### Informe de cada modelo clásico/lineal (matriz de confusión + ejemplos)

Se genera un PNG completo por modelo en `reportes\informes_v2\` (classification_report +
matriz de confusión + grilla de ejemplos) y se muestran en pantalla todas las matrices juntas
para comparar de un vistazo.


In [ ]:
for k, v in sorted(RESULTADOS_ML.items()):
    informe_visual(k, v["test"], modelo=None, n_imgs=6, inline=False, gradcam_n=0)

ml_sorted = sorted(RESULTADOS_ML.items(), key=lambda kv: -kv[1]["test"]["f1_macro"])
n = len(ml_sorted)
ncols = 5
nrows = int(math.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.9 * nrows), squeeze=False)
for j, (k, v) in enumerate(ml_sorted):
    ax = axes[j // ncols][j % ncols]
    cm = confusion_matrix(v["test"]["y_true"], v["test"]["y_pred"],
                          labels=np.arange(NUM_CLASSES))
    with np.errstate(invalid="ignore", divide="ignore"):
        cmn = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
    ax.set_title(f"{k}\n{v['metodo'][:38]}\nmacro-F1 = {v['test']['f1_macro']:.4f}", fontsize=8)
    ax.set_xlabel("predicha"); ax.set_ylabel("real")
for j in range(n, nrows * ncols):
    axes[j // ncols][j % ncols].axis("off")
fig.suptitle("Clasificadores sobre embeddings - matrices normalizadas (orden por F1 test)",
             fontsize=13)
plt.tight_layout(); _guardar("todas_matriz_clasicos")

tabla_ml = pd.DataFrame([dict(ID=k, Metodo=v["metodo"],
                              F1_val=v["val"]["f1_macro"], F1_test=v["test"]["f1_macro"],
                              BalAcc=v["test"]["balanced_acc"],
                              ROC_AUC=v["test"].get("roc_auc"),
                              PR_AUC=v["test"].get("pr_auc"))
                         for k, v in ml_sorted])
display(tabla_ml.round(4))


## Fase 8 — Ítem 6: explicabilidad (LIME + Grad-CAM) y sesgo de dominio

Correcciones respecto de la v1:

- **1000 muestras** por explicación (no 500) y **15 imágenes** (3 por clase, no 10).
- Se explica **la clase real y la predicha** (en los errores importa la real).
- Se **cuantifica** qué fracción de la importancia cae **dentro de la hoja**, usando la misma
  máscara verde del EDA (HSV: tono 30-110°, saturación ≥ 50, valor ≥ 60) → la conclusión se
  escribe **después** de ver los números.
- Se agrega **Grad-CAM** (implementación propia con hooks) sobre las mismas imágenes.
- El F1 por fuente con ~57 imágenes de 2019 era ruido → ahora se entrena un **clasificador de
  `source` sobre los embeddings**: si supera el baseline de mayoría, el embedding sí captura el
  dominio.


In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

MODELOS_LIME = [(k, MODELOS[k]) for k in ["FT_noaug", "FT_aug"]]

muestra_lime = (pd.concat([g.sample(n=min(LIME_POR_CLASE, len(g)), random_state=SEED)
                          for _, g in df_test.groupby("label")])
                .sort_values(["label", "image_id"]).reset_index(drop=True))
print("imagenes a explicar:", len(muestra_lime), "| LIME samples:", LIME_SAMPLES)
display(muestra_lime[["image_id", "label_name", "source"]])

def mascara_verde(pil_img):
    hsv = np.asarray(pil_img.convert("HSV"), dtype=np.int16)
    H = hsv[..., 0] * 360.0 / 255.0
    S, V = hsv[..., 1], hsv[..., 2]
    return ((H >= 30) & (H <= 110) & (S >= 50) & (V >= 60))

def hacer_fn(modelo):
    def fn(imgs):
        arr = np.asarray(imgs, dtype=np.float32)
        if arr.ndim == 3:
            arr = arr[None]
        if arr.max() > 1.5:
            arr = arr / 255.0
        t = torch.from_numpy(arr).permute(0, 3, 1, 2).to(DEVICE)
        mean = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)
        std = torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)
        with torch.no_grad():
            p = torch.softmax(modelo((t - mean) / std), dim=1)
        return p.cpu().numpy()
    return fn

FN = {n: hacer_fn(m) for n, m in MODELOS_LIME}
explainer = lime_image.LimeImageExplainer()

def peso_por_pixel(explanation, label):
    w = explanation.local_exp.get(label, [])
    seg = explanation.segments
    out = np.zeros(seg.shape, dtype=np.float32)
    for sid, val in w:
        out[seg == sid] = val
    return out

def fraccion_en_hoja(pixel_w, hoja):
    m = pixel_w > 0
    if m.sum() == 0 or hoja.sum() == 0:
        return float("nan")
    return float((m & hoja).sum() / m.sum())


In [ ]:
filas_lime = []
t0 = time.time()

for i, row in muestra_lime.iterrows():
    pil = Image.open(DIR_IMG / row["image_id"]).convert("RGB")
    vis = T.Resize((SIZE, SIZE))(pil)
    arr = np.asarray(vis, dtype=np.float32) / 255.0
    hoja = mascara_verde(vis)
    y_real = int(row["label"])

    for nom_modelo, modelo in MODELOS_LIME:
        fn = FN[nom_modelo]
        p = fn(arr[None])[0]
        y_pred = int(p.argmax())
        exp = explainer.explain_instance(arr, fn, top_labels=NUM_CLASSES, hide_color=0,
                                         num_samples=LIME_SAMPLES, random_seed=SEED)
        f_real = fraccion_en_hoja(peso_por_pixel(exp, y_real), hoja)
        f_pred = fraccion_en_hoja(peso_por_pixel(exp, y_pred), hoja)
        filas_lime.append(dict(image_id=row["image_id"], clase_real=CLASS_NAMES[y_real],
                               fuente=str(row["source"]), modelo=nom_modelo,
                               predicha=CLASS_NAMES[y_pred],
                               prob_real=round(float(p[y_real]), 3),
                               prob_pred=round(float(p[y_pred]), 3),
                               acierto=bool(y_pred == y_real),
                               pct_area_hoja=round(float(hoja.mean()), 3),
                               imp_dentro_hoja_real=(None if np.isnan(f_real)
                                                     else round(f_real, 3)),
                               imp_dentro_hoja_pred=(None if np.isnan(f_pred)
                                                     else round(f_pred, 3))))

        fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
        axes[0].imshow(arr); axes[0].imshow(hoja, cmap="Greens", alpha=0.30)
        axes[0].set_title(f"original + mascara hoja\n({100 * hoja.mean():.0f}% del area)")
        for a, (lab, tit) in enumerate([(y_real, "LIME clase REAL"),
                                        (y_pred, "LIME clase PREDICHA")], start=1):
            temp, mask = exp.get_image_and_mask(lab, positive_only=False, num_features=10,
                                                hide_rest=False)
            axes[a].imshow(mark_boundaries(temp, mask))
            axes[a].set_title(f"{tit} -> {CLASS_NAMES[lab]} (p={p[lab]:.2f})")
        axes[3].bar(CLASS_NAMES, p,
                    color=["#D73027", "#FC8D59", "#FCC163", "#4575B4", "#1A9850"])
        axes[3].set_ylim(0, 1); axes[3].set_title("probabilidades")
        axes[3].tick_params(axis="x", rotation=45)
        for a in (0, 1, 2):
            axes[a].axis("off")
        fig.suptitle(f"{row['image_id']} | real={CLASS_NAMES[y_real]} | "
                     f"{nom_modelo} predice {CLASS_NAMES[y_pred]}", fontsize=11)
        plt.tight_layout()
        plt.savefig(DIR_LIME / f"lime_{i:02d}_{row['label_name']}_{nom_modelo}.png",
                    dpi=110, bbox_inches="tight")
        plt.show()

df_lime = pd.DataFrame(filas_lime)
print(f"LIME: {len(df_lime)} explicaciones en {time.time() - t0:.0f}s")
display(df_lime)


### 8.1 ¿Dónde mira el modelo? importancia dentro de la hoja (LIME + Grad-CAM)

La celda anterior ya cuantificó LIME. Acá se agrega Grad-CAM y se resumen **los números**;
la conclusión se calcula desde ellos, no está escrita de antemano.


In [ ]:
filas_cam = []
hoja_media = float(df_lime["pct_area_hoja"].mean())
for i, row in muestra_lime.iterrows():
    pil = Image.open(DIR_IMG / row["image_id"]).convert("RGB")
    vis = T.Resize((SIZE, SIZE))(pil)
    hoja = mascara_verde(vis)
    y_real = int(row["label"])
    x = tf_eval(vis).unsqueeze(0).to(DEVICE)
    for nom_modelo, modelo in MODELOS_LIME:
        with torch.no_grad():
            p = torch.softmax(modelo(x), 1)[0].cpu().numpy()
        y_pred = int(p.argmax())
        for lab, tag in [(y_real, "real"), (y_pred, "predicha")]:
            cam = gradcam(modelo, x, clase=lab)
            den = float(cam.sum())
            frac = float((cam * hoja).sum() / den) if den > 0 else float("nan")
            filas_cam.append(dict(image_id=row["image_id"], modelo=nom_modelo,
                                  clase=CLASS_NAMES[lab], tipo=tag,
                                  imp_dentro_hoja=round(frac, 3),
                                  pct_area_hoja=round(float(hoja.mean()), 3)))
        if i < 2:
            fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
            axes[0].imshow(vis)
            axes[0].set_title(f"{row['image_id']} | real {CLASS_NAMES[y_real]}")
            for a, (lab, tag) in enumerate([(y_real, "REAL"), (y_pred, "PREDICHA")], start=1):
                cam = gradcam(modelo, x, clase=lab)
                axes[a].imshow(vis); axes[a].imshow(cam, cmap="jet", alpha=0.45)
                axes[a].set_title(f"Grad-CAM {tag}: {CLASS_NAMES[lab]}")
            for a in axes:
                a.axis("off")
            fig.suptitle(f"Grad-CAM - {nom_modelo}")
            plt.tight_layout()
            plt.savefig(DIR_LIME / f"gradcam_{i:02d}_{row['label_name']}_{nom_modelo}.png",
                        dpi=110, bbox_inches="tight")
            plt.show()

df_cam = pd.DataFrame(filas_cam)
display(df_cam)

resumen_exp = dict(
    pct_area_hoja_promedio=round(hoja_media, 3),
    lime_imp_dentro_hoja_real=round(float(df_lime["imp_dentro_hoja_real"].mean()), 3),
    lime_imp_dentro_hoja_pred=round(float(df_lime["imp_dentro_hoja_pred"].mean()), 3),
    gradcam_imp_dentro_hoja_real=round(float(df_cam[df_cam.tipo == "real"]
                                             ["imp_dentro_hoja"].mean()), 3),
    gradcam_imp_dentro_hoja_pred=round(float(df_cam[df_cam.tipo == "predicha"]
                                             ["imp_dentro_hoja"].mean()), 3),
)
display(pd.Series(resumen_exp, name="resumen").to_frame())

_ref = resumen_exp["pct_area_hoja_promedio"]
print("=== CONCLUSION GENERADA DESDE LOS NUMEROS ===")
print(f"- La hoja ocupa {_ref:.1%} del cuadro en promedio (eso seria el azar).")
for k in ["lime_imp_dentro_hoja_real", "lime_imp_dentro_hoja_pred",
          "gradcam_imp_dentro_hoja_real", "gradcam_imp_dentro_hoja_pred"]:
    v = resumen_exp[k]
    rel = (v / _ref) if _ref > 0 else float("nan")
    print(f"- {k}: {v:.1%} dentro de la hoja -> {rel:.2f}x el azar "
          f"({'concentrado en hoja' if rel > 1 else 'NO concentrado en hoja'})")
concl_exp = ("LIME y Grad-CAM concentran la importancia dentro de la hoja "
             if min(resumen_exp["lime_imp_dentro_hoja_real"],
                    resumen_exp["gradcam_imp_dentro_hoja_real"]) > _ref
             else "hay importancia fuera de la hoja: revisar (posible sesgo espurio)")
print("CONCLUSION:", concl_exp)


### 8.2 ¿El embedding captura el dominio 2019/2020?

En vez de comparar F1 por fuente con ~57 imágenes de 2019 (ruido), se entrena un clasificador
que **predice `source` desde los embeddings**: si supera el baseline de mayoría, el modelo
sí está leyendo el dominio de captura (y eso es un sesgo potencial).


In [ ]:
src_map = {}
for sp in ["train", "val", "test"]:
    v = manifest[manifest.split == sp]["source"].astype(int).values
    src_map[sp] = (v == 2019).astype(int)      # 1 = 2019, 0 = 2020

filas_src = []
for nombre, emb in [("mv2_384", EMB["mv2"]), ("b7_600", EMB["b7"])]:
    Xtr, _ = emb["train"]; Xv, _ = emb["val"]; Xt, _ = emb["test"]
    ytr, yv, yt = src_map["train"], src_map["val"], src_map["test"]
    if len(np.unique(ytr)) < 2:
        filas_src.append(dict(embeddings=nombre, baseline_mayoria=np.nan,
                              acc_train=np.nan, acc_val=np.nan, acc_test=np.nan,
                              n_2019_test=int(yt.sum()), n_2020_test=int((1 - yt).sum())))
        continue
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=2000, random_state=SEED)
    clf.fit(sc.transform(Xtr), ytr)
    filas_src.append(dict(embeddings=nombre,
                          baseline_mayoria=round(float(max(ytr.mean(), 1 - ytr.mean())), 4),
                          acc_train=round(float(clf.score(sc.transform(Xtr), ytr)), 4),
                          acc_val=round(float(clf.score(sc.transform(Xv), yv)), 4),
                          acc_test=round(float(clf.score(sc.transform(Xt), yt)), 4),
                          n_2019_test=int(yt.sum()), n_2020_test=int((1 - yt).sum())))
tabla_src = pd.DataFrame(filas_src)
display(tabla_src)
if tabla_src["acc_test"].notna().any():
    print("Si acc_test >> baseline, el embedding si captura el dominio de captura (2019 vs 2020),")
    print("aunque nunca se haya usado `source` ni `image_id` como feature.")
else:
    print("Modo smoke: el submuestreo dejó una sola fuente en train; se omite el clasificador de dominio.")


## Fase 9 — Ítem 8: robustez, calibración y abstención

- **Curvas F1 vs severidad** ante los problemas típicos de captura en campo:
  desenfoque (blur σ), ruido, brillo (±10/20/30 %, lo que el EDA prometía) y recorte
  (hoja parcialmente fuera del cuadro).
- **Calibración**: ECE en 10 bins + *temperature scaling* ajustado sobre **val**.
- **Umbral de abstención**: en móvil es muy razonable decir "no sé, tomá otra foto".
  El umbral se elige sobre **val** y se reporta sobre **test**.
- **Clase de rechazo/OOD**: las 33 imágenes "no-hojas" que la v1 borró (32 son CBSD de
  tubérculo, un síntoma real) ahora se usan para medir cuántas detecta la abstención.


In [ ]:
def cargar_test_bruto():
    arr = [np.asarray(T.Resize((SIZE, SIZE))(Image.open(DIR_IMG / f).convert("RGB")),
                      dtype=np.uint8) for f in df_test["image_id"]]
    return np.stack(arr), df_test["label"].astype(int).values

def perturbar(img_u8, tipo, nivel, rng):
    if tipo == "blur":
        if nivel <= 0: return img_u8
        from scipy.ndimage import gaussian_filter
        out = gaussian_filter(img_u8.astype(np.float32), sigma=(nivel, nivel, 0))
        return np.clip(out, 0, 255).astype(np.uint8)
    if tipo == "ruido":
        if nivel <= 0: return img_u8
        out = img_u8.astype(np.float32) + rng.normal(0.0, 255.0 * nivel, img_u8.shape)
        return np.clip(out, 0, 255).astype(np.uint8)
    if tipo == "brillo":
        if nivel == 0: return img_u8
        out = img_u8.astype(np.float32) * (1.0 + float(nivel) / 100.0)
        return np.clip(out, 0, 255).astype(np.uint8)
    if tipo == "recorte":
        if nivel >= 1.0: return img_u8
        h, w, _ = img_u8.shape
        ch, cw = max(8, int(h * nivel)), max(8, int(w * nivel))
        y0, x0 = (h - ch) // 2, (w - cw) // 2
        crop = img_u8[y0:y0 + ch, x0:x0 + cw]
        return np.asarray(Image.fromarray(crop).resize((w, h), Image.BILINEAR), dtype=np.uint8)
    return img_u8

def predecir_u8(modelo, imgs_u8, batch=64):
    out = []
    with torch.no_grad():
        for i in range(0, len(imgs_u8), batch):
            xs = torch.stack([tf_eval(Image.fromarray(im)) for im in imgs_u8[i:i + batch]])
            logits = modelo(xs.to(DEVICE, non_blocking=True)
                            .to(memory_format=torch.channels_last))
            out.append(torch.softmax(logits, 1).cpu().numpy())
    return np.concatenate(out)

TEST_U8, TEST_Y = cargar_test_bruto()
print("test en memoria:", TEST_U8.shape)

def es_base(tipo, lvl):
    if tipo == "recorte":
        return lvl >= 1.0
    if tipo == "brillo":
        return lvl == 0
    return lvl == 0.0

filas_rob = []
for nom_modelo, modelo in MODELOS_LIME:
    print("== robustez:", nom_modelo, "==")
    p_base = predecir_u8(modelo, TEST_U8)
    for tipo, niveles in SEV.items():
        for lvl in niveles:
            if es_base(tipo, lvl):
                p = p_base
            else:
                rng = np.random.default_rng(SEED)
                imgs = np.stack([perturbar(im, tipo, lvl, rng) for im in TEST_U8])
                p = predecir_u8(modelo, imgs)
            m = armar_metricas(TEST_Y, p.argmax(1), p)
            filas_rob.append(dict(modelo=nom_modelo, tipo=tipo, severidad=lvl,
                                  f1_macro=m["f1_macro"], balanced_acc=m["balanced_acc"]))
            print(f"   {tipo:9s} {lvl:>6} -> F1 {m['f1_macro']:.4f}")

df_rob = pd.DataFrame(filas_rob)
display(df_rob.round(4))
df_rob.to_csv(DIR_ROB / ("f1_vs_severidad" + TAG + ".csv"), index=False)

fig, axes = plt.subplots(1, 4, figsize=(21, 4.4))
for ax, tipo in zip(axes, ["blur", "ruido", "brillo", "recorte"]):
    for nom_modelo, _ in MODELOS_LIME:
        d = df_rob[(df_rob.tipo == tipo) & (df_rob.modelo == nom_modelo)]
        ax.plot(d["severidad"], d["f1_macro"], marker="o", label=nom_modelo)
    ax.set_title(f"F1 vs {tipo}"); ax.set_xlabel("severidad")
    ax.set_ylabel("macro-F1 test"); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend()
plt.suptitle("Robustez: macro-F1 bajo perturbaciones de captura")
plt.tight_layout(); plt.show()


In [ ]:
def ece(y_true, y_prob, n_bins=10):
    conf = y_prob.max(axis=1)
    ok = (y_prob.argmax(axis=1) == np.asarray(y_true)).astype(float)
    bordes = np.linspace(0, 1, n_bins + 1)
    total, e = len(conf), 0.0
    for b in range(n_bins):
        m = ((conf > bordes[b]) & (conf <= bordes[b + 1])) if b else \
            ((conf >= bordes[b]) & (conf <= bordes[b + 1]))
        if m.sum():
            e += (m.sum() / total) * abs(ok[m].mean() - conf[m].mean())
    return float(e)

def ajustar_temperatura(y_true_v, prob_v):
    from scipy.optimize import minimize_scalar
    y = np.asarray(y_true_v); P = np.asarray(prob_v)
    def f(T):
        q = np.exp(np.log(P + 1e-12) / T)
        q = q / q.sum(axis=1, keepdims=True)
        return log_loss(y, q, labels=np.arange(NUM_CLASSES))
    r = minimize_scalar(f, bounds=(0.25, 10.0), method="bounded")
    return float(r.x)

def calibrar(y_true_v, prob_v, y_true_t, prob_t, nombre):
    T = ajustar_temperatura(y_true_v, prob_v)
    def aplicar(P, T):
        q = np.exp(np.log(P + 1e-12) / T)
        return q / q.sum(axis=1, keepdims=True)
    return dict(ID=nombre, T=round(T, 3),
                ECE_val_antes=round(ece(y_true_v, prob_v), 4),
                ECE_val_despues=round(ece(y_true_v, aplicar(prob_v, T)), 4),
                ECE_test_antes=round(ece(y_true_t, prob_t), 4),
                ECE_test_despues=round(ece(y_true_t, aplicar(prob_t, T)), 4))

filas_cal = []
for nom_modelo, _ in MODELOS_LIME:
    r = RESULTADOS[nom_modelo]
    filas_cal.append(calibrar(np.array(RESULTADOS[nom_modelo]["val"]["y_true"]),
                              np.array(RESULTADOS[nom_modelo]["val"]["y_prob"]),
                              np.array(r["test"]["y_true"]),
                              np.array(r["test"]["y_prob"]), nom_modelo))
tabla_cal = pd.DataFrame(filas_cal)
display(tabla_cal)

fig, axes = plt.subplots(1, len(filas_cal), figsize=(5.5 * len(filas_cal), 4.6), squeeze=False)
for ax, fila in zip(axes[0], filas_cal):
    r = RESULTADOS[fila["ID"]]
    y = np.array(r["test"]["y_true"]); P = np.array(r["test"]["y_prob"])
    conf = P.max(1); ok = (P.argmax(1) == y)
    bordes = np.linspace(0, 1, 11)
    cx, cy, cw = [], [], []
    for b in range(10):
        m = ((conf > bordes[b]) & (conf <= bordes[b + 1])) if b else \
            ((conf >= bordes[b]) & (conf <= bordes[b + 1]))
        if m.sum():
            cx.append(conf[m].mean()); cy.append(ok[m].mean()); cw.append(m.sum())
    ax.plot([0, 1], [0, 1], ls="--", color="gray")
    ax.scatter(cx, cy, s=[max(30, w) for w in cw], color="tab:blue")
    ax.set_xlabel("confianza"); ax.set_ylabel("precision")
    ax.set_title(f"{fila['ID']} - reliability\nECE test = {fila['ECE_test_antes']:.4f}")
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
def curva_abstencion(y_true, prob, taus):
    filas = []
    for tau in taus:
        acc = prob.max(1) >= tau
        cov = float(acc.mean())
        if acc.sum() == 0:
            filas.append(dict(tau=tau, cobertura=0.0, precision=None,
                              f1_aceptadas=None, costo=None)); continue
        yt, yp = np.asarray(y_true)[acc], prob[acc].argmax(1)
        filas.append(dict(tau=round(float(tau), 2), cobertura=round(cov, 4),
                          precision=round(float((yt == yp).mean()), 4),
                          f1_aceptadas=round(float(f1_score(yt, yp, average="macro",
                                                            labels=np.arange(NUM_CLASSES),
                                                            zero_division=0)), 4),
                          costo=round(costo_realizado(yt, yp), 4)))
    return pd.DataFrame(filas)

taus = np.arange(0.0, 0.951, 0.05)
cv_val = curva_abstencion(RESULTADOS[mejor_ft]["val"]["y_true"],
                          np.array(RESULTADOS[mejor_ft]["val"]["y_prob"]), taus)
cv_test = curva_abstencion(RESULTADOS[mejor_ft]["test"]["y_true"],
                           np.array(RESULTADOS[mejor_ft]["test"]["y_prob"]), taus)
display(pd.concat([cv_val.assign(set="val"), cv_test.assign(set="test")],
                  ignore_index=True).round(4))

valid = cv_val[cv_val["precision"].notna()]
candidatos = valid[valid["precision"] >= 0.85]
TAU = float(candidatos["tau"].max()) if len(candidatos) else 0.0
print(f"Umbral elegido sobre VAL (mayor cobertura con precision >= 0.85): tau = {TAU:.2f}")

r_t = cv_test[cv_test["tau"] == TAU]
if len(r_t):
    r_t = r_t.iloc[0]
    print(f"TEST con tau={TAU:.2f}: cobertura {r_t['cobertura']:.3f} | "
          f"precision de las aceptadas {r_t['precision']:.3f} | "
          f"macro-F1 aceptadas {r_t['f1_aceptadas']:.3f} | costo {r_t['costo']:.3f}")
else:
    print("TEST: tau fuera del rango evaluado")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
axes[0].plot(cv_val["cobertura"], cv_val["precision"], marker="o", label="val")
axes[0].plot(cv_test["cobertura"], cv_test["precision"], marker="o", label="test")
axes[0].axvline(float(cv_test[cv_test["tau"] == TAU]["cobertura"].iloc[0])
                if len(cv_test[cv_test["tau"] == TAU]) else 0, ls=":", color="gray")
axes[0].set_xlabel("cobertura (fraccion aceptada)"); axes[0].set_ylabel("precision")
axes[0].set_title("curva riesgo-cobertura del umbral de abstencion"); axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(cv_test["tau"], cv_test["cobertura"], marker="o", label="cobertura")
axes[1].plot(cv_test["tau"], cv_test["precision"], marker="o", label="precision")
axes[1].axvline(TAU, ls="--", color="tab:red", label=f"tau={TAU:.2f}")
axes[1].set_xlabel("tau (minima probabilidad aceptada)"); axes[1].set_ylim(0, 1)
axes[1].set_title("test: cobertura y precision vs tau"); axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# --- clase de rechazo / OOD: las 33 "no-hojas" ---
if DIR_NOH.exists():
    ids_out = sorted([f for f in os.listdir(DIR_NOH) if f.lower().endswith(".jpg")])
    if ids_out:
        arr_out = np.stack([np.asarray(T.Resize((SIZE, SIZE))(
            Image.open(DIR_NOH / f).convert("RGB")), dtype=np.uint8) for f in ids_out])
        P_out = predecir_u8(MODELOS[mejor_ft], arr_out)
        conf_out = P_out.max(1)
        print(f"\nOOD: {len(ids_out)} imagenes 'no-hojas' (32 CBSD de tuberculo + 1 sana)")
        print(f"  confianza media = {conf_out.mean():.3f} | mediana = {np.median(conf_out):.3f}")
        print(f"  rechazadas con tau={TAU:.2f}: {(conf_out < TAU).sum()}/{len(ids_out)} "
              f"= {(conf_out < TAU).mean():.1%}")
        display(pd.DataFrame({"confianza": conf_out}).describe().round(3))
        print("Leer esto como: con tau actual, cuantas 'no-hojas' habria que volver a fotografiar.")
else:
    print("no existe la carpeta de no-hojas")


## Fase 10 — Estadística del test: intervalos de confianza

El test tiene **300 imágenes (60 por clase)**: con eso el macro-F1 tiene un IC amplio, así que
comparar modelos por diferencias de 1-2 puntos es ruido. Se usa **bootstrap (10 000 remuestreos)**
para el IC 95 % de cada modelo y **bootstrap pareado** para la diferencia entre dos modelos.
Además, el mejor modelo se evalúa también sobre **`test_pool_rest` (752 imágenes)**.


In [ ]:
def bootstrap_f1(y_true, y_pred, n=BOOT_N, seed=SEED):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    N = len(y_true)
    rng = np.random.default_rng(seed)
    vals = np.empty(n, dtype=np.float64)
    for b in range(n):
        idx = rng.integers(0, N, N)
        vals[b] = f1_score(y_true[idx], y_pred[idx], average="macro",
                           labels=np.arange(NUM_CLASSES), zero_division=0)
    return vals

def bootstrap_delta(ra, rb, n=BOOT_N, seed=SEED):
    ya = np.asarray(ra["y_true"]); pa = np.asarray(ra["y_pred"])
    yb = np.asarray(rb["y_true"]); pb = np.asarray(rb["y_pred"])
    assert np.array_equal(ya, yb), "los modelos deben compartir test para compararlos"
    N = len(ya)
    rng = np.random.default_rng(seed)
    vals = np.empty(n, dtype=np.float64)
    for b in range(n):
        idx = rng.integers(0, N, N)
        vals[b] = (f1_score(yb[idx], pb[idx], average="macro",
                            labels=np.arange(NUM_CLASSES), zero_division=0)
                   - f1_score(ya[idx], pa[idx], average="macro",
                              labels=np.arange(NUM_CLASSES), zero_division=0))
    return vals

def agregar_ic(nombre, res, fila):
    v = bootstrap_f1(res["test"]["y_true"], res["test"]["y_pred"])
    fila["IC95_lo"] = round(float(np.percentile(v, 2.5)), 4)
    fila["IC95_hi"] = round(float(np.percentile(v, 97.5)), 4)
    return fila

todos_ic = dict(RESULTADOS)
if res_antes is not None:
    todos_ic["E1 (v1)"] = res_antes
todos_ic.update(RESULTADOS_ML)

filas_ic = []
for k, v in todos_ic.items():
    f = dict(ID=k, F1_test=v["test"]["f1_macro"])
    agregar_ic(k, v, f)
    filas_ic.append(f)
tabla_ic = pd.DataFrame(filas_ic).sort_values("F1_test", ascending=False)
display(tabla_ic.round(4))

pares = [(mejor_ft, "FT_aug" if mejor_ft == "FT_noaug" else "FT_noaug"),
         (mejor_ft, "E1 (v1)")]
filas_pares = []
for a, b in pares:
    if a not in todos_ic or b not in todos_ic:
        continue
    ya = np.asarray(todos_ic[a]["test"]["y_true"])
    yb = np.asarray(todos_ic[b]["test"]["y_true"])
    if not np.array_equal(ya, yb):
        print(f"[skip] {b} vs {a}: y_true no coincide ({len(ya)} vs {len(yb)}), "
              "no se compara (diferentes corridas/subsets)")
        continue
    d = bootstrap_delta(todos_ic[a]["test"], todos_ic[b]["test"])
    filas_pares.append(dict(comparacion=f"{b} -> {a}",
                            delta_f1=round(float(d.mean()), 4),
                            IC95_lo=round(float(np.percentile(d, 2.5)), 4),
                            IC95_hi=round(float(np.percentile(d, 97.5)), 4),
                            significativo=bool(np.percentile(d, 2.5) > 0)))
display(pd.DataFrame(filas_pares).round(4))
df_pares = pd.DataFrame(filas_pares)
print("Si el IC de la diferencia incluye 0, la mejora no es distinguible del ruido.")

# --- segunda evaluacion: test_pool_rest (752 imagenes held-out) ---
pool = manifest[manifest.split == "test_pool_rest"].reset_index(drop=True)
if len(pool) and mejor_ft in MODELOS:
    ds_pool = MandiocaDataset(manifest, "test_pool_rest", tf_eval)
    r_pool = evaluar_completo(MODELOS[mejor_ft], ds_pool, batch=64)
    print(f"\n{mejor_ft} sobre test_pool_rest ({len(pool)} imgs): "
          f"macro-F1 = {r_pool['f1_macro']:.4f} | BalAcc = {r_pool['balanced_acc']:.4f} | "
          f"ROC-AUC = {r_pool.get('roc_auc')}")
else:
    r_pool = None
    print("test_pool_rest vacio o sin modelo")


## Fase 11 — Consolidado: CSV, resumen en `00_resumen_consigna_y_avance.md` y reproducibilidad

Última fase: todo lo medido se vuelca a **un solo CSV** (`reportes\03_resultados_v2.csv`,
sección por sección) y el **resumen del proyecto** recibe la fila de resultados v2 entre los
marcadores `<!-- INICIO-RESULTADOS-V2 -->` / `<!-- FIN-RESULTADOS-V2 -->`.


In [ ]:
def _seccion(nombre, dfx):
    if dfx is None or not isinstance(dfx, pd.DataFrame) or len(dfx) == 0:
        return None
    d = dfx.copy()
    d.insert(0, "seccion", nombre)
    return d

partes = [
    _seccion("01_fine_tuning", tabla_ab),
    _seccion("02_clasicos_embeddings", tabla_ml),
    _seccion("03_ic_bootstrap", tabla_ic),
    _seccion("04_costos_decision", tabla_costos),
    _seccion("05_costo_computacional", tabla_costo),
    _seccion("06_calibracion", tabla_cal),
    _seccion("07_abstencion_val",
             cv_val.assign(set="val") if "cv_val" in dir() else None),
    _seccion("07_abstencion_test",
             cv_test.assign(set="test") if "cv_test" in dir() else None),
    _seccion("08_robustez", df_rob if "df_rob" in dir() else None),
    _seccion("09_limite_source", tabla_src),
    _seccion("10_lime_resumen",
             pd.DataFrame([resumen_exp]) if "resumen_exp" in dir() else None),
    _seccion("11_lime_detalle", df_lime),
    _seccion("12_gradcam", df_cam),
    _seccion("13_bootstrap_diferencias", df_pares if "df_pares" in dir() else None),
]
partes = [p for p in partes if p is not None]
df_csv = pd.concat(partes, ignore_index=True, sort=False)
df_csv.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")
print("CSV:", CSV_OUT, "|", len(df_csv), "filas |",
      df_csv["seccion"].nunique(), "secciones")
display(df_csv.groupby("seccion").size().to_frame("filas"))


In [ ]:
ruta_md = DIR_REPORT / "00_resumen_consigna_y_avance.md"
try:
    txt = ruta_md.read_text(encoding="utf-8")
except UnicodeDecodeError:
    txt = ruta_md.read_text(encoding="cp1252")

def _f(x, nd=4):
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return "-"
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

mejor = RESULTADOS[mejor_ft]
row_mejor = tabla_ab[tabla_ab["ID"] == mejor_ft].iloc[0]
row_e1 = tabla_ab[tabla_ab["ID"] == "E1 (v1)"]
ic_mejor = tabla_ic[tabla_ic["ID"] == mejor_ft].iloc[0]
costo_mejor = tabla_costos[tabla_costos["ID"] == mejor_ft].iloc[0]
lat_mejor = tabla_costo[tabla_costo["ID"] == mejor_ft].iloc[0]
cal_mejor = tabla_cal[tabla_cal["ID"] == mejor_ft].iloc[0]
rob_min = df_rob[df_rob.modelo == mejor_ft].sort_values("f1_macro").iloc[0]
lin = []

lin.append("### 6.1 — Resultados v2 (`02b_transfer_learning_mandioca_v2.ipynb`)\n")
lin.append(f"**Modo:** {'SMOKE (validación rápida)' if SMOKE else 'FULL'} · "
           f"**seed** {SEED} · **input** {SIZE} px · **test** 300 imgs (60/clase) · "
           f"**selección por val**.\n")
lin.append("| ID | Método | F1-val | F1-test | IC 95 % (bootstrap) | Costo/test | ms/img |")
lin.append("|---|---|---|---|---|---|---|")
for _, r in tabla_ab.iterrows():
    ic = tabla_ic[tabla_ic["ID"] == r["ID"]]
    co = tabla_costos[tabla_costos["ID"] == r["ID"]]
    la = tabla_costo[tabla_costo["ID"] == r["ID"]]
    ic_s = (f"[{_f(ic.iloc[0]['IC95_lo'])}, {_f(ic.iloc[0]['IC95_hi'])}]"
            if len(ic) else "-")
    lin.append(f"| {r['ID']} | {r['Metodo']} | {_f(r['F1_val'])} | {_f(r['F1_test'])} | "
               f"{ic_s} | {_f(co.iloc[0]['costo_realizado_argmax'], 3) if len(co) else '-'} | "
               f"{_f(la.iloc[0]['ms_img_batch1'], 1) if len(la) else '-'} |")
lin.append("")
lin.append(f"- **Mejor FT (elegido por val):** `{mejor_ft}` → **macro-F1 test = "
           f"{_f(row_mejor['F1_test'])}** (val {_f(row_mejor['F1_val'])}), "
           f"costo {_f(costo_mejor['costo_realizado_argmax'], 3)}, "
           f"{_f(lat_mejor['ms_img_batch1'], 1)} ms/img, "
           f"{_f(lat_mejor['GFLOPs'], 2)} GFLOPs, {_f(lat_mejor['MB_ckpt'], 1)} MB.")
if len(row_e1):
    r0 = row_e1.iloc[0]
    lin.append(f"- **vs. v1 (E1, backbone congelado):** {_f(r0['F1_test'])} → "
               f"{_f(row_mejor['F1_test'])} "
               f"(Δ = {float(row_mejor['F1_test']) - float(r0['F1_test']):+.4f}).")
if len(df_pares):
    for _, p in df_pares.iterrows():
        lin.append(f"- **Δ bootstrap** ({p['comparacion']}): {p['delta_f1']:+.4f} "
                   f"IC95 [{p['IC95_lo']:+.4f}, {p['IC95_hi']:+.4f}] → "
                   f"{'significativa' if p['significativo'] else 'NO distinguible del ruido'}.")
lin.append(f"- **Calibración:** ECE test {_f(cal_mejor['ECE_test_antes'])} → "
           f"{_f(cal_mejor['ECE_test_despues'])} con temperatura "
           f"T = {_f(cal_mejor['T'], 2)}.")
if "TAU" in dir():
    lin.append(f"- **Abstención:** tau = {TAU:.2f} (elegido por val, precisión ≥ 0.85); "
               f"ver tabla 07 en el CSV para cobertura/precisión en test y rechazo OOD.")
if "resumen_exp" in dir():
    lin.append(f"- **Explicabilidad:** la hoja ocupa {resumen_exp['pct_area_hoja_promedio']:.1%} "
               f"del cuadro; LIME concentra {resumen_exp['lime_imp_dentro_hoja_real']:.1%} "
               f"y Grad-CAM {resumen_exp['gradcam_imp_dentro_hoja_real']:.1%} de la importancia "
               f"dentro de la hoja (clase real).")
if len(df_rob):
    lin.append(f"- **Robustez (peor caso medido para {mejor_ft}):** {rob_min['tipo']} "
               f"nivel {rob_min['severidad']} → macro-F1 {_f(rob_min['f1_macro'])} "
               f"(base {_f(df_rob[(df_rob.modelo == mejor_ft) & (df_rob.severidad.isin([0, 0.0, 1.0]))]['f1_macro'].max())}).")
if len(tabla_src) and pd.notna(tabla_src["acc_test"]).any():
    ts = tabla_src.iloc[0]
    cap = bool(ts['acc_test'] > ts['baseline_mayoria'] + 0.05)
    lin.append(f"- **Dominio (2019 vs 2020) desde embeddings:** acc test "
               f"{_f(ts['acc_test'])} vs baseline {_f(ts['baseline_mayoria'])} "
               f"→ el embedding {'sí' if cap else 'NO'} "
               f"captura el dominio de captura.")
lin.append("")
lin.append(f"Detalle completo: `reportes\\03_resultados_v2.csv` (secciones 01-13), "
           f"informes en `reportes\\informes_v2\\`, LIME/Grad-CAM en "
           f"`reportes\\lime_gradcam_v2\\`, robustez en `reportes\\robustez_v2\\`.\n")

bloque = ("<!-- INICIO-RESULTADOS-V2 -->\n" + "\n".join(lin) +
          "\n<!-- FIN-RESULTADOS-V2 -->")

if "<!-- INICIO-RESULTADOS-V2 -->" in txt:
    pre, post = txt.split("<!-- INICIO-RESULTADOS-V2 -->", 1)
    _, post = post.split("<!-- FIN-RESULTADOS-V2 -->", 1)
    txt = pre + bloque + post
elif "<!-- FIN-RESULTADOS-TL -->" in txt:
    i = txt.index("<!-- FIN-RESULTADOS-TL -->") + len("<!-- FIN-RESULTADOS-TL -->")
    txt = txt[:i] + "\n\n" + bloque + txt[i:]
elif "\n## 7." in txt:
    i = txt.index("\n## 7.")
    txt = txt[:i] + "\n" + bloque + "\n" + txt[i:]
else:
    txt = txt.rstrip() + "\n\n" + bloque + "\n"

try:
    ruta_md.write_text(txt, encoding="utf-8")
    print("markdown actualizado:", ruta_md)
except Exception as e:
    print("NO se pudo actualizar el .md:", e)


### Reproducibilidad

Semilla única (42) en `random`/`numpy`/`torch`/`cuda`, `cudnn.deterministic = True` y
`benchmark = False`; `num_workers = 0` en smoke (2 en full) con generator sembrado; los pesos de
clase se calculan **solo sobre train**; la selección de hiperparámetros y del modelo se hace por
**val**; el test se reporta una sola vez por corrida. Cada experimento deja JSON + `.pth` en
`modelos\v2\` con sufijo `_smoke` cuando corresponde, así una corrida rápida **no pisa** la full.


In [ ]:
info = dict(
    fecha=time.strftime("%Y-%m-%d %H:%M:%S"),
    python=sys.version.split()[0],
    torch=torch.__version__, timm=timm.__version__, numpy=np.__version__,
    pandas=pd.__version__, device=str(DEVICE),
    gpu=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"),
    smoke=SMOKE, seed=SEED, size=SIZE, num_classes=NUM_CLASSES,
    n_train=int((manifest.split == "train").sum()),
    n_val=int((manifest.split == "val").sum()),
    n_test=int((manifest.split == "test").sum()),
    n_test_pool_rest=int((manifest.split == "test_pool_rest").sum()),
    mejor_ft=mejor_ft,
    mejor_ft_test_f1=round(float(RESULTADOS[mejor_ft]["test"]["f1_macro"]), 4),
    e1_v1_test_f1=(round(float(res_antes["test"]["f1_macro"]), 4)
                   if res_antes is not None else None),
    total_seg_desde_inicio=round(time.time() - T_INICIO, 1),
)
ruta_info = DIR_V2 / ("run_info" + TAG + ".json")
ruta_info.write_text(json.dumps(info, indent=1, ensure_ascii=False), encoding="utf-8")
print(json.dumps(info, indent=1, ensure_ascii=False))
print("->", ruta_info)

import subprocess
def _n(p):
    return len([x for x in Path(p).glob("*") if x.is_file()]) if Path(p).exists() else 0
print("\nArtefactos generados:")
print("  modelos/v2/              :", _n(DIR_V2))
print("  reportes/informes_v2/    :", _n(DIR_INFORMES))
print("  reportes/lime_gradcam_v2/:", _n(DIR_LIME))
print("  reportes/robustez_v2/    :", _n(DIR_ROB))
print("  CSV                      :", CSV_OUT.exists(), "-", CSV_OUT.name)

assert CSV_OUT.exists(), "falta el CSV consolidado"
assert RESULTADOS[mejor_ft]["test"]["f1_macro"] > 0, "F1 del mejor FT inválido"
print("\nOK: corrida v2 completa sin errores.")
